# SuSiNE Functionality Tutorial


In [ ]:
library(devtools)
load_all()

# utils::data("SuSiE_N3_X", package = "susine")
data("N3finemapping") #This comes from the SuSiE pkg under 'data/'N3finemapping.rData'
attach(N3finemapping)

library(susieR)
head(X)

In [ ]:
# Randomly select 5 causal SNPs, by index
set.seed(1)
n = nrow(X)
p = ncol(X)
causal_idx = sample(1:ncol(X), 5)
print(paste0("Causal SNP indices: ", paste(causal_idx, collapse = ", ")))


#Randomly generate effect sizes for these SNPs
b = rep(0, p)
b[causal_idx] = rnorm(n=5, mean = 0, sd = 1)
print(paste0("Causal SNP effect sizes: ", paste(round(b[causal_idx],2), collapse = ", ")))

# Generate phenotype vector
y_noise = 0.5 #level of noise in y as a percentage of the variance of y
sigma_2 = var(X %*% b) * (y_noise / (1 - y_noise))
y = X %*% b + rnorm(n, mean = 0, sd = sqrt(sigma_2))

In [ ]:
# Create a vector of SNP annotations (prior means)
mu_0 = rnorm(n=p, mean = 0, sd = 1)
mu_0[causal_idx] = b[causal_idx] + rnorm(n=5, mean = 0, sd = 0.1) #make the prior means close to the true effect sizes for the causal SNPs

# Create a vector of SNP annotations (prior variances)
sigma_0_2 = mu_0^2 #set prior variances to be proportional to the square of the prior means 

In [ ]:
#Run susine
fit = susine(L=5, X, y, prior_update_method = "none") #No EB updates

# Show SNPs with highest PIPs, order descending, rounded to 2 decimals, including the name of the index
pips = fit$model_fit$PIPs
sorted_pips_res = sort(pips, decreasing = TRUE, index.return = TRUE)
top_pips = head(sorted_pips_res$x, 10)
top_indices = head(sorted_pips_res$ix, 10)
pips_df = data.frame(
  SNP_Index = top_indices,
  PIP = round(top_pips, 2)
)
pips_df$True_Selection = as.integer(pips_df$SNP_Index %in% causal_idx)
print("SNPs with highest PIPs:")
pips_df



In [ ]:
# ---
# **Metrics and Helper Functions**
#
# Before we start fitting models, let's define some functions to evaluate their performance.
# We will calculate:
# 1.  **Predicted SNP Heritability (h_g^2)**: How much of the trait's variance the model attributes to genetics.
# 2.  **PR-AUC**: Area under the Precision-Recall curve, which measures how well the model identifies the true causal SNPs.
# 3.  **PIP Cross-Entropy**: A measure of how accurate the model's Posterior Inclusion Probabilities (PIPs) are. Lower is better.

if (!requireNamespace("PRROC", quietly = TRUE)) install.packages("PRROC")
library(PRROC)

# Create a binary vector indicating the true causal SNPs
truth <- integer(p)
truth[causal_idx] <- 1L

# Function to calculate PR-AUC
pr_auc_vec <- function(scores, truth01) {
  pr <- PRROC::pr.curve(scores.class0 = scores[truth01 == 1], scores.class1 = scores[truth01 == 0], curve = FALSE)
  return(pr$auc.davis.goadrich)
}

# Function to calculate PIP Cross-Entropy
cross_entropy <- function(pip, truth01) {
  eps <- .Machine$double.eps
  pip <- pmin(pmax(pip, eps), 1 - eps)
  -mean(truth01 * log(pip) + (1 - truth01) * log(1 - pip))
}

# Calculate the true heritability from our simulated data
h2_true <- var(as.vector(X %*% b)) / var(y)
cat(sprintf("True SNP Heritability (h_g^2): %.3f\n", h2_true))

# Function to predict heritability from a susine fit
h2_pred_from_fit <- function(fit) {
  s2 <- tail(fit$model_fit$sigma_2, 1)
  1 - s2 / var(y)
}

# A simple function to print the metrics for a fit
report_metrics <- function(fit) {
    pips <- fit$model_fit$PIPs
    cat(sprintf("Predicted h_g^2: %.3f\n", h2_pred_from_fit(fit)))
    cat(sprintf("PR-AUC: %.3f\n", pr_auc_vec(pips, truth)))
    cat(sprintf("PIP Cross-Entropy: %.3f\n", cross_entropy(pips, truth)))
}

---
# **Part A: No User Priors**
In this section, we'll explore scenarios where we don't provide any specific prior information about the effect sizes (`mu_0`) or their variances (`sigma_0_2`). We will let `susine` either use naive defaults or learn these parameters from the data using Empirical Bayes (EB).

In [ ]:
### **a.i. SuSiE - Naïve sigma, naïve mu**
#
# This is the most basic case. We set `prior_update_method = "none"`, which means the model will use its default, non-informative priors for both the effect mean (`mu_0 = 0`) and variance (`sigma_0_2` derived from `var(y)`), with no Empirical Bayes updates. This is equivalent to a standard SuSiE analysis.

cat("--- Running: a.i. SuSiE (naive mu, naive sigma; no EB) ---\n")
fit_a_i <- susine(L=5, X, y, prior_update_method = "none")
test <- susie(X, y, L=5,estimate_prior_variance = FALSE)


# fit_a_i$model_fit$PIPs
# test$pip

#Find which PIPs differ the most between the two implementations
pip_diffs  = test$pip -fit_a_i$model_fit$PIPs
max_diff_idx = which(pip_diffs == max(pip_diffs))
paste("SNP with largest PIP difference:", max_diff_idx)
paste("SuSiE value:", round(test$pip[max_diff_idx],4))
paste("SuSiNE value:",  round(fit_a_i$model_fit$PIPs[max_diff_idx],4))
paste("Difference:", round(pip_diffs[max_diff_idx],4))


In [ ]:
### **a.ii. SuSiE - EB sigma, naïve mu**
#
# Here, we keep the prior mean at zero (`mu_0 = 0`) but allow the model to learn the prior variance of the effects from the data. We do this by setting `prior_update_method = "var"`. This can help the model better estimate the magnitude of the true causal effects.

cat("\n--- Running: a.ii. SuSiE (naive mu, EB sigma) ---\n")
fit_a_ii <- susine(L=5, X, y, prior_update_method = "var")
test <- susie(X, y, L=5,estimate_prior_variance = TRUE)

#Find which PIPs differ the most between the two implementations
pip_diffs  = test$pip -fit_a_ii$model_fit$PIPs
max_diff_idx = which(pip_diffs == max(pip_diffs))
paste("SNP with largest PIP difference:", max_diff_idx)
paste("SuSiE value:", round(test$pip[max_diff_idx],4))
paste("SuSiNE value:",  round(fit_a_ii$model_fit$PIPs[max_diff_idx],4))
paste("Difference:", round(pip_diffs[max_diff_idx],4))

test$V
fit_a_ii$priors$sigma_0_2[,1]

In [ ]:
### **a.iii. SuSiNE - EB mu, naïve sigma**
#
# Now we introduce SuSiNE. We allow the model to learn a single, shared prior mean for all SNPs by setting `prior_update_method = "mean"`. The prior variance remains fixed at its default. This is useful if you suspect a general directional effect but don't have SNP-specific information.

cat("\n--- Running: a.iii. SuSiNE (EB mu, naive sigma) ---\n")
fit_a_iii <- susine(L=5, X, y, prior_update_method = "mean")
report_metrics(fit_a_iii)

In [ ]:
### **a.iv. SuSiNE - EB mu, EB sigma**
#
# This is the most flexible of the "no user prior" cases. By setting `prior_update_method = "both"`, we allow the model to learn both a shared prior mean and a prior variance from the data.

cat("\n--- Running: a.iv. SuSiNE (EB mu, EB sigma) ---\n")
fit_a_iv <- susine(L=5, X, y, prior_update_method = "both")
report_metrics(fit_a_iv)

---
# **Part B: User Priors**
Now, let's see how we can improve performance by providing the model with informative priors based on our simulated annotations. The `mu_0` and `sigma_0_2` vectors we created earlier will be used as these "functional" priors.

In [ ]:
### **b.i. SuSiE + functional sigma**
#
# In this case, we still assume the prior mean is zero (`mu_0 = 0`), but we provide a prior for the variance that is derived from our annotations. We'll use `sigma_0_2 = mu_0^2` as a simple way to specify that SNPs with larger prior mean annotations are expected to have larger effect size variance. We set `pum = "none"` to use these priors without modification.

cat("\n--- Running: b.i. SuSiE + functional sigma (fixed) ---\n")
# Note: sigma_0_2 is interpreted as a proportion of var(y)
functional_sigma <- mu_0^2 / var(y)
fit_b_i <- susine(L=5, X, y, mu_0 = 0, sigma_0_2 = functional_sigma, prior_update_method = "none")
report_metrics(fit_b_i)
fit_b_i$priors

In [ ]:
### **b.ii. SuSiNE + functional mu (scale_var option)**
#
# Here, we provide our annotation-derived `mu_0` vector. We set `prior_update_method = "scale"`, which tells `susine` to trust the *relative pattern* of our `mu_0` vector but to learn an optimal scaling factor for it, as well as learning the prior variance. This is a powerful way to leverage imperfect prior information.

cat("\n--- Running: b.ii. SuSiNE + functional mu (scale + EB sigma) ---\n")
fit_b_ii <- susine(L=5, X, y, mu_0 = mu_0, prior_update_method = "scale")
report_metrics(fit_b_ii)
fit_b_ii$priors

In [ ]:
### **b.iii. SuSiNE - functional mu, functional sigma**
#
# For this final case, we provide the model with our best possible prior information for both `mu_0` and `sigma_0_2`, and we tell it to use them without any EB updates (`pum = "none"`). This simulates a scenario where we have high confidence in our prior annotations. We'll estimate a good `sigma_0_2` externally by looking at the difference between our `mu_0` annotation and the true effects `b` at the causal SNPs.

cat("\n--- Running: b.iii. SuSiNE (functional mu & sigma; fixed) ---\n")

# Estimate sigma_0_2 externally using ground truth (for demonstration)
# This is the mean squared difference between the true effects and our prior means at causal SNPs
sigma0_sq_hat <- mean((b[causal_idx] - mu_0[causal_idx])^2)
# Convert to proportion of Var(y) for the model
sigma0_prop_hat <- as.numeric(sigma0_sq_hat / var(y))

fit_b_iii <- susine(L=5, X, y, mu_0 = mu_0, sigma_0_2 = rep(sigma0_prop_hat, p), prior_update_method = "none")
report_metrics(fit_b_iii)